# FULL-SYN — vòng 1

[Plan FULL-SYN](../../docs/project/full-syn-train-plan-2026-10-22.md), mục 4.1 và 5. Notebook này là **log của vòng 1**:
cell output ghi cấu hình, hash, kết quả từng trạm và quyết định ở mỗi cổng; log đầy đủ của code
(checkpoint, `train.log`, `step_metrics.jsonl`, predictions, policy, bản notebook đã chạy) nằm trong campaign
dưới `data/training/`. Vòng chỉnh sau là **file mới** (`05_full_syn_r2.ipynb`) với campaign và settings mới,
để giữ lịch sử đối chứng. Bị ngắt giữa chừng thì chạy lại chính file này: bước đã xong được bỏ qua,
task đang dở chạy tiếp từ `last.ckpt`.

Các cổng tự dừng notebook (cell báo `GateStop`):

| Cổng | Dừng khi |
|---|---|
| Sau G1 (D18) | val cAcc-M2 của FSA < 52,8% |
| Sau G2 (D17, D19) | 🟡/🔴 ở vòng 1–2; vòng 3: 🔴 (🟡 chạy tiếp, ghi "kỳ vọng ngang B2") |
| Trước khi train task 2 | precision nhãn giả của model task 1 trên ảnh task 2 < 0,5 |
| G4 | chỉ chấm test một lần; đã có kết quả thì in lại, không chấm lại |

In [1]:
import json, os, sys
from pathlib import Path
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file())
sys.path.insert(0, str(REPO))
from autocheckout import full_syn
from autocheckout.io import load_json
from autocheckout.training_workflow import resolve
SETTINGS_PATH = resolve(os.environ.get('AUTOCHECKOUT_TRAINING_SETTINGS', 'data/training/settings_full_syn_r1.json'))
SETTINGS = load_json(SETTINGS_PATH)
PLAN = full_syn.prepare(SETTINGS)
print('Settings:', SETTINGS_PATH)
print('Campaign:', PLAN['root'], '| round', PLAN['round'], '| dry run' if PLAN['dry_run'] else '| real run')
print('Synthetic config sha256:', PLAN['release_config_sha256'])
print('Train images per task:', PLAN['train_images'])
print(json.dumps({k: PLAN[k] for k in ('resolution', 'batch_size', 'effective_batch', 'fsa_batch_size', 'num_workers', 'tf32',
                                         'fsa_epochs', 'epochs', 'audit_images')}, indent=1))

Settings: /workspace/AutoCheckout-CL/data/training/full-syn-r1/notebook_executions/05_full_syn_r1/78ef03d9-7ef0-4c41-a8bc-d6602602cce8/settings.json
Campaign: /workspace/AutoCheckout-CL/data/training/full-syn-r1 | round 1 | real run
Synthetic config sha256: cbd29ff0b63eb8aa93fdc48aef6d8eef0ca7116ac7cdc67bc3531fa6c5896eca
Train images per task: [21752, 12534, 12129, 11850, 13377]
{
 "resolution": 640,
 "batch_size": 8,
 "effective_batch": 8,
 "fsa_batch_size": 4,
 "num_workers": 8,
 "tf32": true,
 "fsa_epochs": 8,
 "epochs": 16,
 "audit_images": 1000
}


In [2]:
import platform, subprocess, torch
from autocheckout.runinfo import git_state
git = git_state(REPO)
print('git', git.get('commit'), '| dirty' if git.get('dirty') else '| clean')
print('Python', platform.python_version(), '| torch', torch.__version__, '| CUDA', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout if torch.cuda.is_available() else '')

git ad240e8a280881311480a95196f35bbec523b5f4 | dirty
Python 3.10.22 | torch 2.2.2+cu121 | CUDA 12.1
GPU: NVIDIA GeForce RTX 4090
name, memory.total [MiB], driver_version
NVIDIA GeForce RTX 4090, 24564 MiB, 550.144.03



## G1 — FSA và kiểm tra sớm (D18)

Fine-tune toàn bộ trên ảnh ghép task 1, tối đa `fsa_epochs`; xuất `hf_model` từ epoch có AP50 val_task_1 cao nhất.

In [3]:
FSA = full_syn.stage_fsa(PLAN)
print('Epoch đã chọn:', FSA['epoch_selection']['epoch'], '| AP50 val:', [round(c['AP50'], 4) for c in FSA['epoch_selection']['curve']])
print(json.dumps({k: FSA[k] for k in ('cAcc_m2', 'cAcc_m1_single', 'AP50', 'cAcc_by_level', 'train_hours', 'gate')}, indent=1))

2026-10-26T12:44:03+07:00 START bash /workspace/AutoCheckout-CL/scripts/run_exp.sh /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN-FSA/resolved_config.sh -> /workspace/AutoCheckout-CL/data/training/full-syn-r1/attempts/fsa/c10baf17-3575-4752-b4ae-7c748f6a8d88/EXP-FULL-SYN-FSA_task1-1_val_runner.log
== EXP-FULL-SYN-FSA: 1 tasks in /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN-FSA (2026-10-26 05:44:03)
-- task 1: training (2026-10-26 05:44:03)
/venv/pdp/lib/python3.10/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Seed set to 0
Multi-scale deformable attention: CUDA kernel
loading annotations into memory...
Done (t=0.15s)
creating index...
index created!
loading annotations

## G2 — PDP task 1 từ FSA và cổng D17

In [4]:
TASK1 = full_syn.stage_task1(PLAN)
print(json.dumps({k: TASK1[k] for k in ('cAcc_m2', 'cAcc_m1_single', 'threshold_single', 'AP50', 'cAcc_by_level',
                                        'train_hours', 'gate')}, indent=1))
print('Nhầm lẫn nhiều nhất:', TASK1['confusions_top20'][:10])

2026-10-26T17:47:09+07:00 START bash /workspace/AutoCheckout-CL/scripts/run_exp.sh /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN/resolved_config.sh -> /workspace/AutoCheckout-CL/data/training/full-syn-r1/attempts/task1/e6dc894d-6ee4-425f-bcf1-298a1bedbac1/EXP-FULL-SYN_task1-1_val_runner.log
== EXP-FULL-SYN: 1 tasks in /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN (2026-10-26 10:47:09)
-- task 1: training (2026-10-26 10:47:09)
/venv/pdp/lib/python3.10/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Seed set to 0
Multi-scale deformable attention: CUDA kernel
loading annotations into memory...
Done (t=0.16s)
creating index...
index created!
loading annotations into memory..

## G3 — task 2–5

Trước mỗi task t: audit nhãn giả của model task t−1 trên `audit_images` ảnh ghép của task t, so với
`train_task_<t>_gt_full.json`. Sau mỗi task: chấm val, ngưỡng theo nhóm task, ghi `task_<t>/task_log.json`.

In [5]:
TASKS = full_syn.stage_tasks(PLAN)
for t, r in sorted(TASKS['tasks'].items(), key=lambda item: int(item[0])):
    p = r['pseudo_labels']
    print(f"task {t}: cAcc M1 {r['cAcc_m1_groups']:.4f} (nhóm {r['thresholds_groups']}) / {r['cAcc_m1_single']:.4f} (một ngưỡng)"
          f" | AP50 {r['AP50']} | nhãn giả P {p['all'].get('precision')} R {p['recall']} | I3/F14 {p['filters']}"
          f" | {r['train_hours']} h")

2026-10-26T23:45:16+07:00 START bash -ec 'source "$CONFIG"; cd "$REPO/pdp"; exec "$PYTHON" ppg_audit.py "${ARGS[@]}" "$@"' ppg_audit --output_dir /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN --audit_task 2 --audit_images 1000 --prev_ckpt /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN/task_1/task_final.pth --task_ann_dir /workspace/AutoCheckout-CL/data/processed/rpc_synth_v2_v1/tasks/synth_v2 -> /workspace/AutoCheckout-CL/data/training/full-syn-r1/attempts/task2/d53c3a1f-941b-4880-9b1c-d509040be272/ppg_audit_task2_runner.log
/venv/pdp/lib/python3.10/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
loading annotations into memory...
Done (t=1.23s)
creating index...
index cr

## G4 — chấm test một lần

In [6]:
FINAL = full_syn.stage_final_test(PLAN)
print((Path(PLAN['root']) / 'SUMMARY.md').read_text())

2026-10-27T21:39:28+07:00 START bash /workspace/AutoCheckout-CL/scripts/run_exp.sh /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN/resolved_config.sh -> /workspace/AutoCheckout-CL/data/training/full-syn-r1/attempts/final_test/1eed5e8a-890e-45ad-b6d6-e928c4faf403/EXP-FULL-SYN_task1-5_test_runner.log
== EXP-FULL-SYN: 5 tasks in /workspace/AutoCheckout-CL/data/training/full-syn-r1/runs/EXP-FULL-SYN (2026-10-27 14:39:28)
-- task 1: trained, rewriting predictions
/venv/pdp/lib/python3.10/site-packages/lightning_utilities/core/imports.py:14: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Seed set to 0
Multi-scale deformable attention: CUDA kernel
loading annotations into memory...
Done (t=0.14s)
creating index...
index created!
Setting task id :  0
loadin